# 🌌 Deepfake Detection AI — Google Colab GPU Fine-Tuning

This notebook guides you through fine-tuning the **InceptionResnetV1** spatial face model and **Temporal Video Sequence Model** on Google Colab's free GPU (Tesla T4 / V100 / A100).

### ⚡ Quick Start:
1. In the top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU** (or any GPU) > Save.
2. Run the cells step-by-step from top to bottom.

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive (Recommended)
Mounting Google Drive allows you to:
- Load large datasets directly from your Drive without re-uploading every time
- Save trained model checkpoints permanently so they won't be lost when the Colab session disconnects

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository & Setup Environment
Clones the codebase and installs all necessary deep learning dependencies.

In [ ]:
# Clone repo or change directory
import os
if not os.path.exists('Deepfake-Detection'):
    !git clone https://github.com/Chamansengar/Deepfake-Detection-Using-Ai-Frontend-.git Deepfake-Detection

%cd Deepfake-Detection

In [ ]:
# Install dependencies required for model training
%pip install -q facenet-pytorch opencv-python-headless scikit-learn torchvision tqdm

## 4. Rapid Pipeline Test (Dry Run)
Run a 2-epoch test using synthetic dummy data to confirm that PyTorch, GPU acceleration, data augmentation, and checkpoint saving are all working properly.

In [ ]:
!python train.py --dummy_run --epochs 2 --batch_size 8

## 5. Prepare Your Dataset

Place your training images into the following directory structure:
```
data/
  ├── real/   <- authentic face photos (jpg, png, webp)
  └── fake/   <- deepfake / synthetic face photos
```

If your dataset is compressed as a `.zip` in Google Drive, uncompress it directly here:

In [ ]:
# Example: Unzip dataset from Google Drive
# !unzip -q '/content/drive/MyDrive/deepfake_dataset.zip' -d ./data

# Or check if data directory exists
!ls -la ./data

## 6. Fine-Tune the Spatial Model (InceptionResnetV1)

This runs fine-tuning on the GPU. Key options:
- `--data_dir`: Path to folder containing `real/` and `fake/` subdirectories (automatically splits 80% train / 20% validation).
- `--epochs`: Total training passes (e.g. 20-30).
- `--batch_size`: Batch size (32 or 64 on T4 GPU).
- `--lr`: Learning rate (1e-4 recommended).
- `--resume`: (Optional) Resume from an existing checkpoint e.g. `./checkpoints/best_model.pth`.

In [ ]:
!python train.py \
    --data_dir ./data \
    --epochs 25 \
    --batch_size 32 \
    --lr 0.0001 \
    --checkpoint_dir ./checkpoints

## 7. Save & Download Fine-Tuned Checkpoint

Copy the new `best_model.pth` directly to Google Drive or trigger a browser download to place it back into your local app's `checkpoints/` folder.

In [ ]:
# Option A: Copy to Google Drive
!cp ./checkpoints/best_model.pth /content/drive/MyDrive/best_model.pth
print("✓ Saved to Google Drive: MyDrive/best_model.pth")

# Option B: Direct browser download
from google.colab import files
files.download('./checkpoints/best_model.pth')

## 8. (Optional) Fine-Tune Temporal Video Sequence Model
If you have raw video clips (.mp4, .avi) in `data/videos/real` and `data/videos/fake`, you can fine-tune the Transformer temporal model:

In [ ]:
!python train_temporal.py \
    --video_dir ./data/videos \
    --epochs 30 \
    --seq_len 16 \
    --temporal_head transformer \
    --checkpoint_dir ./checkpoints